In [2]:
from __future__ import annotations

import logging
from pathlib import Path
from typing import Tuple

import joblib
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

DATASET_PATH = Path(r"D:\Python-Course\ML-Course\test\breast_cancer_dataset.csv")
MODEL_PATH = Path(r"D:\Python-Course\ML-Course\test\Machine-Learning-Algorithm\SVM\svm_model.pkl")

TARGET_COLUMN = "target"

TEST_SIZE = 0.20
RANDOM_STATE = 42

SVM_CONFIG = {
    "kernel": "rbf",
    "C": 1.0,
    "gamma": "scale",
    "random_state": RANDOM_STATE,
}


# ---------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

logger = logging.getLogger(__name__)


# ---------------------------------------------------------------------
# Data Loading
# ---------------------------------------------------------------------

def load_dataset(filepath: Path) -> pd.DataFrame:
    """
    Load dataset from CSV.

    Args:
        filepath: CSV file path.

    Returns:
        DataFrame
    """
    if not filepath.exists():
        raise FileNotFoundError(f"Dataset not found: {filepath}")

    logger.info("Loading dataset from %s", filepath)

    return pd.read_csv(filepath)


def split_dataset(
    data: pd.DataFrame,
    target_column: str,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """
    Split dataset into train and test sets.
    """

    if target_column not in data.columns:
        raise ValueError(f"Target column '{target_column}' not found.")

    X = data.drop(columns=[target_column])
    y = data[target_column]

    return train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )


# ---------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------

def build_pipeline() -> Pipeline:
    """
    Create ML pipeline.
    """

    pipeline = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("classifier", SVC(**SVM_CONFIG)),
        ]
    )

    return pipeline


def evaluate_model(model: Pipeline, X_test, y_test) -> None:
    """
    Evaluate trained model.
    """

    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    logger.info("Model Accuracy : %.4f", accuracy)

    print("\nAccuracy")
    print("-" * 30)
    print(f"{accuracy:.4f}")

    print("\nConfusion Matrix")
    print("-" * 30)
    print(confusion_matrix(y_test, predictions))

    print("\nClassification Report")
    print("-" * 30)
    print(classification_report(y_test, predictions))


# ---------------------------------------------------------------------
# Persistence
# ---------------------------------------------------------------------

def save_model(model: Pipeline, filepath: Path) -> None:
    """
    Save trained model.
    """
    joblib.dump(model, filepath)
    logger.info("Model saved to %s", filepath)


# ---------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------

def main() -> None:

    try:
        data = load_dataset(DATASET_PATH)

        X_train, X_test, y_train, y_test = split_dataset(
            data,
            TARGET_COLUMN,
        )

        model = build_pipeline()

        logger.info("Training model...")

        model.fit(X_train, y_train)

        logger.info("Training completed.")

        evaluate_model(model, X_test, y_test)

        save_model(model, MODEL_PATH)

    except Exception:
        logger.exception("Application failed.")
        raise


if __name__ == "__main__":
    main()

2026-06-30 16:15:04,968 | INFO | Loading dataset from D:\Python-Course\ML-Course\test\breast_cancer_dataset.csv
2026-06-30 16:15:05,050 | INFO | Training model...
2026-06-30 16:15:05,075 | INFO | Training completed.
2026-06-30 16:15:05,089 | INFO | Model Accuracy : 0.9825



Accuracy
------------------------------
0.9825

Confusion Matrix
------------------------------
[[41  1]
 [ 1 71]]

Classification Report
------------------------------
              precision    recall  f1-score   support

           0       0.98      0.98      0.98        42
           1       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



2026-06-30 16:15:05,424 | INFO | Model saved to D:\Python-Course\ML-Course\test\Machine-Learning-Algorithm\SVM\svm_model.pkl
